# 00 · Inventario de datasets

Alcance: archivos, tamaños, filas, schemas, nulos, cardinalidades, IDs y fechas candidatas. Lectura completa; no hay estimaciones ni EDA.

In [1]:
from pathlib import Path
import sys
root = Path.cwd()
if not (root / "AGENTS.md").exists() and (root.parent / "AGENTS.md").exists():
    root = root.parent
assert (root / "AGENTS.md").exists(), "Ejecutar desde raíz o notebooks/"
sys.path.insert(0, str(root))
import pandas as pd
from IPython.display import display
from src.config import REPORTS
pd.set_option("display.max_rows", 100)


## 1. Descubrimiento, integridad e inventario

Se inspeccionan todos los encabezados y se calcula SHA-256 antes y después. Los CSV originales permanecen donde están. DuckDB procesa una tabla a la vez con límite de 2 GB y derrame temporal a disco. Los campos vacíos se interpretan como NULL; no se imputan. Se excluyen columnas de partición Hive y se preserva la ruta del archivo para contar registros correctamente, incluso en CSV multilínea.

In [2]:
from src.profiling_utils import inventory
summaries, columns = inventory()
datasets = pd.DataFrame(summaries)
datasets['MiB'] = datasets['bytes'] / 2**20
display(datasets)

Inventario: branches


Inventario: call_center_interactions


Inventario: call_transcripts


Inventario: campaign_sends


Inventario: complaints


...

Inventario: products


Inventario: satisfaction_surveys


Inventario: service_agents


Inventario: transactions


,dataset,files,rows,columns,bytes,schema_variants,MiB
0,branches,1,350,22,89529,1,0.085382
1,call_center_interactions,1097,686296,21,139734950,1,133.261633
2,call_transcripts,1097,171321,18,137235657,1,130.878121
3,campaign_sends,1083,1746801,22,325976325,1,310.875249
4,complaints,1097,67095,27,17990612,1,17.157185
5,customers,1,150000,27,46897358,1,44.724806
6,daily_exchange_rates,1,13164,7,778914,1,0.742830
7,digital_events,1097,15620994,26,3757355051,1,3583.293010
8,marketing_campaigns,1,200,13,32710,1,0.031195
9,products,1,400000,17,68213646,1,65.053602


## 2. Schemas, nulos y cardinalidades

In [3]:
schema = pd.DataFrame(columns)
display(schema)
print('Archivos:', datasets.files.sum(), 'Filas:', datasets.rows.sum(), 'Bytes:', datasets.bytes.sum())

,dataset,column,physical_type,rows,nulls,null_pct,cardinality,whitespace_only,padded_values,possible_id,possible_date,unique_non_null
0,branches,branch_id,VARCHAR,350,0,0.000000,350,0,0,True,False,True
1,branches,branch_code,VARCHAR,350,0,0.000000,350,0,0,False,False,True
2,branches,branch_name,VARCHAR,350,0,0.000000,175,0,0,False,False,False
3,branches,branch_type,VARCHAR,350,0,0.000000,4,0,0,False,False,False
4,branches,address,VARCHAR,350,0,0.000000,350,0,0,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...
255,transactions,response_code,VARCHAR,4425008,221033,4.995087,5,0,0,False,False,False
256,transactions,is_fraud,VARCHAR,4425008,0,0.000000,2,0,0,False,False,False
257,transactions,fraud_score,VARCHAR,4425008,885157,20.003512,5014,0,0,False,False,False
258,transactions,latitude,VARCHAR,4425008,3567680,80.625391,850555,0,0,False,False,False


Archivos: 7671 Filas: 23495188 Bytes: 5349322481


## 3. Posibles claves y fechas

In [4]:
display(schema[schema.possible_id | schema.unique_non_null])
display(schema[schema.possible_date])

,dataset,column,physical_type,rows,nulls,null_pct,cardinality,whitespace_only,padded_values,possible_id,possible_date,unique_non_null
0,branches,branch_id,VARCHAR,350,0,0.000000,350,0,0,True,False,True
1,branches,branch_code,VARCHAR,350,0,0.000000,350,0,0,False,False,True
4,branches,address,VARCHAR,350,0,0.000000,350,0,0,False,False,True
10,branches,phone,VARCHAR,350,0,0.000000,350,0,0,False,False,True
11,branches,email,VARCHAR,350,0,0.000000,350,0,0,False,False,True
18,branches,latitude,VARCHAR,350,0,0.000000,350,0,0,False,False,True
19,branches,longitude,VARCHAR,350,0,0.000000,350,0,0,False,False,True
22,call_center_interactions,interaction_id,VARCHAR,686296,0,0.000000,686296,0,0,True,False,True
25,call_center_interactions,customer_id,VARCHAR,686296,0,0.000000,148443,0,0,True,False,False
26,call_center_interactions,agent_id,VARCHAR,686296,0,0.000000,1090,0,0,True,False,False


,dataset,column,physical_type,rows,nulls,null_pct,cardinality,whitespace_only,padded_values,possible_id,possible_date,unique_non_null
20,branches,branch_opening_date,VARCHAR,350,0,0.000000,346,0,0,False,True,False
23,call_center_interactions,interaction_date,VARCHAR,686296,0,0.000000,683625,0,0,False,True,False
24,call_center_interactions,process_date,VARCHAR,686296,0,0.000000,1097,0,0,False,True,False
45,call_transcripts,process_date,VARCHAR,171321,0,0.000000,1097,0,0,False,True,False
62,campaign_sends,send_date,VARCHAR,1746801,0,0.000000,1729711,0,0,False,True,False
63,campaign_sends,process_date,VARCHAR,1746801,0,0.000000,1083,0,0,False,True,False
72,campaign_sends,open_date,VARCHAR,1746801,1259492,72.102775,485996,0,0,False,True,False
74,campaign_sends,click_date,VARCHAR,1746801,1649008,94.401595,97749,0,0,False,True,False
77,campaign_sends,conversion_date,VARCHAR,1746801,1737002,99.439032,9798,0,0,False,True,False
84,complaints,creation_date,VARCHAR,67095,0,0.000000,67074,0,0,False,True,False


## 4. Hipótesis de relaciones

Los IDs compartidos y los alias observados (por ejemplo registration_branch_id → branch_id) sugieren relaciones. Su cobertura y la unicidad de los destinos se comprueban en 01; los nombres por sí solos no prueban integridad ni significado de negocio.

In [5]:
from src.profiling_utils import ALIASES, TARGETS
hypotheses = []
for item in columns:
    target_col = ALIASES.get(item['column'], item['column'])
    target = TARGETS.get(target_col)
    if target and target != item['dataset']:
        hypotheses.append({'source': item['dataset'], 'column': item['column'], 'target': target, 'target_column': target_col})
display(pd.DataFrame(hypotheses))

,source,column,target,target_column
0,call_center_interactions,customer_id,customers,customer_id
1,call_center_interactions,agent_id,service_agents,agent_id
2,call_transcripts,interaction_id,call_center_interactions,interaction_id
3,call_transcripts,customer_id,customers,customer_id
4,call_transcripts,agent_id,service_agents,agent_id
5,campaign_sends,campaign_id,marketing_campaigns,campaign_id
6,campaign_sends,customer_id,customers,customer_id
7,complaints,customer_id,customers,customer_id
8,complaints,affected_product_id,products,product_id
9,complaints,related_branch_id,branches,branch_id


## 5. Resultados y límites

Inventario exacto exportado a reports/profiling: datasets.csv, files.csv, inventory_columns.csv, source_manifest.json e inventory_integrity.json. VARCHAR es el tipo de ingesta deliberado; los tipos semánticos se examinan en 01. Una columna única es candidata técnica, no una clave de negocio confirmada. No se imprimen registros individuales de clientes.